# Agent Pipeline for Medical Summaries

**Clean, agent-like flow:** Transcript → Extract Facts → Validate → Generate Summary

**Principles:**
- Simple, readable code
- Clear separation of concerns
- No frameworks, no magic
- Validation before generation

---


In [12]:
import os
import sys
import json
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load environment
backend_dir = Path(os.getcwd()).parent
load_dotenv(backend_dir / '.env')
sys.path.insert(0, str(backend_dir))

# Import prompts and tools
from utils.prompts.extraction_prompt import EXTRACTION_PROMPT
from utils.prompts.summary_from_facts import SUMMARY_FROM_FACTS_PROMPT
from utils.tools.validation_tools import validate_extracted_facts, print_validation_results

print("✓ Environment loaded")
print("✓ Agent pipeline ready")

# Transcribe audio file using Whisper
def transcribe_audio(audio_file):
    """Transcribe audio file using OpenAI Whisper API."""
    print(f"Transcribing {audio_file}...", end=" ")
    
    client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    with open(f'audio/{audio_file}', 'rb') as f:
        response = client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            language="en"
        )
    
    transcript = response.text
    print(f"✅ ({len(transcript)} chars)")
    return transcript

print("✓ Transcribe function ready")


✓ Environment loaded
✓ Agent pipeline ready
✓ Transcribe function ready


In [15]:
# Load sample text transcript
transcript_file = 'sample_visit_1.txt'
with open(f'transcripts/{transcript_file}', 'r') as f:
    transcript = f.read()

print(f'✓ Loaded transcript ({len(transcript)} chars)')
print(f'📄 Preview:{transcript[:2000]}...')

# Alternative: Use audio file instead
# audio_file = '7f935bd1-6976-4fc7-b9a3-08aa59211c34.m4a'
# transcript = transcribe_audio(audio_file)


✓ Loaded transcript (1944 chars)
📄 Preview:Doctor: Good morning! How are you feeling today?

Patient: Hi Doctor. Not so great, to be honest. I've been having this really bad chest pain for about three days now.

Doctor: I see. Can you describe the pain for me? Is it sharp, dull, or something else?

Patient: It's more like a tightness, and it's pretty constant. I'd say it's about a 6 out of 10 in terms of pain.

Doctor: Okay. Does it get worse when you breathe deeply or cough?

Patient: Yes, definitely. When I take a deep breath, it hurts more.

Doctor: Have you had any other symptoms? Fever, cough, shortness of breath?

Patient: I've had a bit of a fever, maybe 100 degrees, and some coughing. The cough started yesterday.

Doctor: Alright. Let me listen to your lungs. [performs examination] I can hear some crackling sounds in your lower left lung. Based on your symptoms and the exam, I think you have mild pneumonia.

Patient: Pneumonia? Is that serious?

Doctor: It can be, but yours is 

## Step 1: Extract Facts

Use LLM to extract explicit facts from transcript.


In [16]:
def extract_facts(transcript: str, model: str = "gpt-4o-mini") -> dict:
    """Extract facts from transcript using LLM."""
    client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    prompt = EXTRACTION_PROMPT.format(transcript=transcript)
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": prompt}],
        temperature=0.1,  # Low temp for consistent extraction
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)


# Extract facts
print("Extracting facts from transcript...")
facts = extract_facts(transcript)

print("✓ Facts extracted\n")
print("Extracted facts:")
print(json.dumps(facts, indent=2))


Extracting facts from transcript...
✓ Facts extracted

Extracted facts:
{
  "symptoms": [
    {
      "description": "chest pain",
      "duration": "about three days",
      "severity": "6 out of 10"
    },
    {
      "description": "fever",
      "duration": "Not mentioned in transcript",
      "severity": "100 degrees"
    },
    {
      "description": "cough",
      "duration": "started yesterday",
      "severity": "Not mentioned in transcript"
    }
  ],
  "diagnosis": {
    "stated_diagnosis": "mild pneumonia",
    "doctor_assessment": "It can be serious, but yours is mild, and we caught it early."
  },
  "medications": [
    {
      "name": "azithromycin",
      "dose": "500 milligrams",
      "timing": "once a day",
      "duration": "five days"
    }
  ],
  "warnings": [
    "Complete the full course of antibiotics, even if you start feeling better before the five days are up.",
    "If your symptoms get worse or you develop difficulty breathing, call me right away or go to 

## Step 2: Validate Facts

Use Python tool to validate extracted facts (no LLM).


In [17]:
# Validate extracted facts
errors = validate_extracted_facts(facts)
is_valid = print_validation_results(errors)

if not is_valid:
    print("\n⚠️  Pipeline stopped due to validation errors")
    print("Fix the extraction prompt or data structure before proceeding.")


✅ Validation passed


## Step 3: Generate Summary

Use LLM to generate patient-friendly summary from validated facts.


In [18]:
def generate_summary_from_facts(facts: dict, model: str = "gpt-4o-mini") -> dict:
    """Generate summary from validated facts using LLM."""
    client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    prompt = SUMMARY_FROM_FACTS_PROMPT.format(
        current_datetime=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        facts_json=json.dumps(facts, indent=2)
    )
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": prompt}],
        temperature=0.2,
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)


# Only proceed if validation passed
if is_valid:
    print("Generating summary from validated facts...")
    summary = generate_summary_from_facts(facts)
    
    print("✓ Summary generated\n")
    
    # Display summary
    print("=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)
    
    print(f"\n📋 Title: {summary.get('title')}")
    
    print(f"\n📝 Summary ({len(summary.get('summary', '').split())} words):")
    print(summary.get('summary'))
    
    print(f"\n🔍 Key Diagnoses:")
    for d in summary.get('key_diagnoses', []):
        print(f"  • {d}")
    
    print(f"\n💊 Medications:")
    for m in summary.get('medications', []):
        print(f"  • {m}")
    
    print(f"\n✅ Action Items:")
    for a in summary.get('action_items', []):
        print(f"  • {a}")
    
    print("\n" + "=" * 80)
else:
    print("\n❌ Skipping summary generation (validation failed)")


Generating summary from validated facts...
✓ Summary generated

FINAL SUMMARY

📋 Title: Pneumonia Follow-Up

📝 Summary (61 words):
The patient has been experiencing chest pain for about three days, rated 6 out of 10, along with a fever of 100 degrees and a cough that started yesterday. The doctor diagnosed the patient with mild pneumonia, noting that it can be serious but was caught early. The patient has been prescribed azithromycin, 500 milligrams once a day for five days.

🔍 Key Diagnoses:
  • mild pneumonia

💊 Medications:
  • azithromycin 500 milligrams once a day for five days

✅ Action Items:
  • Complete the full course of antibiotics, even if feeling better before five days are up.
  • Call the doctor or go to the emergency room if symptoms worsen or difficulty breathing develops.



## Complete Pipeline Function

Run the entire pipeline in one function for easy testing.


In [8]:
def run_pipeline(transcript: str, model: str = "gpt-4o-mini", verbose: bool = True) -> dict:
    """
    Run complete pipeline: Extract → Validate → Summarize
    
    Args:
        transcript: Raw medical visit transcript
        model: LLM model to use
        verbose: Print progress messages
        
    Returns:
        dict with 'facts', 'errors', 'summary' keys
    """
    result = {
        "facts": None,
        "errors": [],
        "summary": None,
        "success": False
    }
    
    # Step 1: Extract
    if verbose:
        print("Step 1/3: Extracting facts...")
    
    facts = extract_facts(transcript, model)
    result["facts"] = facts
    
    if verbose:
        print(f"✓ Extracted facts")
    
    # Step 2: Validate
    if verbose:
        print("\nStep 2/3: Validating facts...")
    
    errors = validate_extracted_facts(facts)
    result["errors"] = errors
    
    if errors:
        if verbose:
            print(f"❌ Validation failed ({len(errors)} errors)")
            for error in errors:
                print(f"  - {error}")
        return result
    
    if verbose:
        print("✓ Validation passed")
    
    # Step 3: Summarize
    if verbose:
        print("\nStep 3/3: Generating summary...")
    
    summary = generate_summary_from_facts(facts, model)
    result["summary"] = summary
    result["success"] = True
    
    if verbose:
        print("✓ Summary generated")
        print("\n" + "=" * 80)
        print("PIPELINE COMPLETE")
        print("=" * 80)
    
    return result


# Test the pipeline
print("Running complete pipeline...\n")
pipeline_result = run_pipeline(transcript)

if pipeline_result["success"]:
    print(f"\n📋 {pipeline_result['summary']['title']}")
    print(f"\n{pipeline_result['summary']['summary']}")
else:
    print("\n❌ Pipeline failed. Check errors above.")


Running complete pipeline...

Step 1/3: Extracting facts...
✓ Extracted facts

Step 2/3: Validating facts...
✓ Validation passed

Step 3/3: Generating summary...
✓ Summary generated

PIPELINE COMPLETE

📋 Headache and Dizziness

The patient has been experiencing a headache for three days and has also reported dizziness. Further details regarding the severity and duration of the dizziness were not provided.
